# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
- Dataset identifier: `10.71728/senscience.vcs2-05nj`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets and their fields, referencing by `@id`.

Each entity (record set, field, column, etc.) is referenced by its unique `@id`.

In [ ]:
# List all record sets and fields by @id
record_sets = []
fields_per_recordset = {}

for recordset in dataset.record_sets():
    record_sets.append(recordset['@id'])
    fields = []
    for field in recordset.get('field', []):
        fields.append(field['@id'])
    fields_per_recordset[recordset['@id']] = fields

# Print overview
print("Record sets found:")
for rs_id in record_sets:
    print(f"- {rs_id}: Fields: {fields_per_recordset[rs_id]}")

### Inspect a record from the primary record set

You can use the `dataset.records(record_set=...)` method, referencing the record set by its `@id`.

In [ ]:
# Example: Show sample records from first record set
if record_sets:
    main_recordset_id = record_sets[0]
    print(f"\nSample records from record set '@id': {main_recordset_id}")
    for i, x in enumerate(dataset.records(record_set=main_recordset_id)):
        print(f"Record {i}: {x}")
        if i >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for recordset_id in record_sets:
    recs = list(dataset.records(record_set=recordset_id))
    # If no records, skip
    if len(recs) == 0:
        continue
    df = pd.DataFrame(recs)
    dataframes[recordset_id] = df
    print(f"DataFrame for '@id': {recordset_id} - Columns: {df.columns.tolist()}")
    print(df.head())
    print("-"*60)

## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data using fields referenced by `@id`. Select a numeric field (e.g., GAD-7 score) and a grouping field (e.g., gender or village).

In [ ]:
# Identify fields for EDA
main_df_id = record_sets[0] if record_sets else None

# Display example columns for reference
if main_df_id is not None and main_df_id in dataframes:
    df = dataframes[main_df_id]
    print(f"Columns available in main DataFrame ({main_df_id}):")
    print(df.columns.tolist())

    # Try likely numeric and grouping fields
    numeric_field_candidates = [col for col in df.columns if 'gad7' in col.lower() or 'phq9' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()]
    group_field_candidates = [col for col in df.columns if 'gender' in col.lower() or 'village' in col.lower() or 'education' in col.lower()]

    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
    group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]

    print(f"Using numeric field: {numeric_field}")
    print(f"Using grouping field: {group_field}")

    # Filtering
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
        norm_col = numeric_field + '_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())
    else:
        print(f"Field {numeric_field} is not numeric and cannot be normalized.")

    # Grouping
    if group_field in filtered_df.columns:
        if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            grouped_df = filtered_df.groupby(group_field).size()
            print(f"Count by {group_field}:")
            print(grouped_df.head())
else:
    print("No relevant DataFrame found for EDA.")

## 5. Visualization
Visualize distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: Histogram of numeric field
if main_df_id is not None and main_df_id in dataframes:
    df = dataframes[main_df_id]
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field], bins=15, kde=True)
        plt.title(f"Distribution of {numeric_field} (Field '@id')")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

    # Boxplot by group field
    if group_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
- The dataset was successfully loaded using the Croissant schema and explored by referencing all entities by their `@id`.
- Records were loaded for each record set, fields were identified and processed.
- Filtered and normalized statistics were explored using numeric survey fields, grouped by key demographic fields, and visualized.
- These steps provide a foundation for more advanced analyses, AI-ready processing, and reproducible mental health insights relevant for Kenya and similar contexts.

For more information and reproducibility, consult the full dataset documentation at [https://sen.science/doi/10.71728/senscience.vcs2-05nj](https://sen.science/doi/10.71728/senscience.vcs2-05nj) and the [mlcroissant documentation](https://mlcroissant.org/) for further processing options and schema exploration.